In [2]:
import numpy as np
import pandas as pd
import yfinance as yf

In [3]:
df=yf.download("TCS.NS",start="2023-01-01",end="2023-12-31",interval="1d")
df.reset_index(inplace=True)


/tmp/ipython-input-85066104.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df=yf.download("TCS.NS",start="2023-01-01",end="2023-12-31",interval="1d")
[*********************100%***********************]  1 of 1 completed


In [9]:
def new_sma(df,window):
 df[f"sma_{window}"] = df["Close"].rolling(window).mean()
 return df

In [10]:
df=new_sma(df,50)
df=new_sma(df,100)

In [14]:
def generate_Signal(df):
  df["Signal"]=0
  df.loc[df["sma_50"]>df["sma_100"],"Signal"]=1
  return df

df=generate_Signal(df)

In [45]:
def backtest(data,signal,initial_capital):

    capital = initial_capital

    stop_loss_pct = 0.02
    take_profit_pct = 0.04

    position = 0
    entry_price = 0.0
    in_trade = False

    trades = []
    equity_curve = []

    for i in range(len(data)):

        price = float(data.iloc[i]["Close"])
        sig = int(signal[i])

        if in_trade:
            sl_price = entry_price * (1 - stop_loss_pct)
            tp_price = entry_price * (1 + take_profit_pct)

            if price <= sl_price or price >= tp_price:
                pnl = position * (price - entry_price)
                capital += position * price
                trades.append(pnl)

                position = 0
                in_trade = False

        if not in_trade and sig == 1:
            qty = int(capital // price)
            if qty > 0:
                position = qty
                entry_price = price
                capital -= qty * price
                in_trade = True

        equity = capital
        if position > 0:
            equity += position * price
        equity_curve.append(equity)

    if in_trade:
        final_price = float(data.iloc[-1]["Close"])
        pnl = position * (final_price - entry_price)
        capital += position * final_price
        trades.append(pnl)

    trades = np.array(trades)
    total_trades = len(trades)
    winning_trades = np.sum(trades > 0)
    losing_trades = np.sum(trades <= 0)

    win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0
    net_profit = capital - initial_capital
    return_pct = (net_profit / initial_capital) * 100

    equity_curve = np.array(equity_curve)
    drawdown = equity_curve / np.maximum.accumulate(equity_curve) - 1
    max_drawdown = drawdown.min()

    returns = np.diff(equity_curve) / equity_curve[:-1]
    sharpe_ratio = np.mean(returns) / np.std(returns) if np.std(returns) != 0 else 0

    print(f"Initial Capital: ₹{initial_capital}")
    print(f"Final Capital: ₹{round(capital, 2)}")
    print(f"Net Profit: ₹{round(net_profit, 2)}")
    print(f"Return: {round(return_pct, 2)}%")
    print(f"Sharpe Ratio: {round(sharpe_ratio, 4)}")
    print(f"Maximum Drawdown: {round(max_drawdown * 100, 2)}%")
    print(f"Total Trades: {total_trades}")
    print(f"Winning Trades: {winning_trades}")
    print(f"Losing Trades: {losing_trades}")
    print(f"Win Rate: {round(win_rate, 2)}%")

    assert winning_trades + losing_trades == total_trades



In [46]:
signal = df["Signal"].values
backtest(df, signal, initial_capital=100000)

Initial Capital: ₹100000
Final Capital: ₹116322.42
Net Profit: ₹16322.42
Return: 16.32%
Sharpe Ratio: 0.0796
Maximum Drawdown: -8.13%
Total Trades: 11
Winning Trades: 6
Losing Trades: 5
Win Rate: 54.55%


/tmp/ipython-input-2205268318.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  price = float(data.iloc[i]["Close"])
/tmp/ipython-input-2205268318.py:46: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  final_price = float(data.iloc[-1]["Close"])


In [47]:
### Q2 SHORT TRADE

In [48]:
def generate_sma_short(df):
   df["Signal"]=0
   df.loc[df["sma_50"]>df["sma_100"],"Signal"]=1
   df.loc[df["sma_50"]<df["sma_100"],"Signal"]=-1

   return df

df=generate_Signal(df)


In [51]:
def short_backtest(data,signal,initial_capital):

    capital = initial_capital

    stop_loss_pct = 0.02
    take_profit_pct = 0.04

    position = 0
    entry_price = 0.0
    in_trade = False

    trades = []
    equity_curve = []

    for i in range(len(data)):

        price = float(data.iloc[i]["Close"])
        sig = int(signal[i])

        if in_trade:
            sl_price = max(sl_price,entry_price * (1 + stop_loss_pct))
            tp_price = max(tp_price,entry_price * (1 - take_profit_pct))

            if price <= sl_price or price >= tp_price:
                pnl = position * (price - entry_price)
                capital += position * price
                trades.append(pnl)

                position = 0
                in_trade = False

        if not in_trade and sig == 1:
            qty = int(capital // price)
            if sig==1 and qty > 0:
                position = qty
                entry_price = price
                capital -= qty * price
                in_trade = True
                sl_price = price * (1 - stop_loss_pct)
                tp_price = price * (1 + take_profit_pct)

            elif sig==-1 and qty<0:
                position = qty
                entry_price = price
                in_trade = True
                sl_price = price * (1 + stop_loss_pct)
                tp_price = price * (1 - take_profit_pct)


        equity = capital
        if position > 0:
            equity += position * price
        elif position<0:
          equity += (-position) * (entry_price - price)
        equity_curve.append(equity)

    if in_trade:
        final_price = float(data.iloc[-1]["Close"])
        if position>0:
         pnl = position * (final_price - entry_price)
         capital += position * final_price
        else:
          qty=-position
          pnl = qty * (entry_price - final_price)
          capital += pnl

        trades.append(pnl)

    trades = np.array(trades)
    total_trades = len(trades)
    winning_trades = np.sum(trades > 0)
    losing_trades = np.sum(trades <= 0)

    win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0
    net_profit = capital - initial_capital
    return_pct = (net_profit / initial_capital) * 100

    equity_curve = np.array(equity_curve)
    drawdown = equity_curve / np.maximum.accumulate(equity_curve) - 1
    max_drawdown = drawdown.min()

    returns = np.diff(equity_curve) / equity_curve[:-1]
    sharpe_ratio = np.mean(returns) / np.std(returns) if np.std(returns) != 0 else 0

    print(f"Initial Capital: ₹{initial_capital}")
    print(f"Final Capital: ₹{round(capital, 2)}")
    print(f"Net Profit: ₹{round(net_profit, 2)}")
    print(f"Return: {round(return_pct, 2)}%")
    print(f"Sharpe Ratio: {round(sharpe_ratio, 4)}")
    print(f"Maximum Drawdown: {round(max_drawdown * 100, 2)}%")
    print(f"Total Trades: {total_trades}")
    print(f"Winning Trades: {winning_trades}")
    print(f"Losing Trades: {losing_trades}")
    print(f"Win Rate: {round(win_rate, 2)}%")

    assert winning_trades + losing_trades == total_trades

In [52]:
short_backtest(df, signal, initial_capital=100000)

Initial Capital: ₹100000
Final Capital: ₹116322.42
Net Profit: ₹16322.42
Return: 16.32%
Sharpe Ratio: 0.0796
Maximum Drawdown: -8.13%
Total Trades: 98
Winning Trades: 49
Losing Trades: 49
Win Rate: 50.0%


/tmp/ipython-input-1787241402.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  price = float(data.iloc[i]["Close"])
/tmp/ipython-input-1787241402.py:58: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  final_price = float(data.iloc[-1]["Close"])
